### 🔗 Links and Resources
- [Partitioning](https://www.databricks.com/discover/pages/optimize-data-workloads-guide#partitioning)

### 📌 Reading data into a Spark DataFrame

In [0]:
df = spark.read.table("population_metrics.default.countries_consolidated")

### 📌 When to Partition
- Partitioning can speed up your queries if you provide the partition column(s) as filters or join on partition column(s) or aggregate on partition column(s) or merge on partition column(s), as it will help Spark to skip a lot of unnecessary data partition (i.e., subfolders) during scan time.
- Databricks recommends not to partition tables under 1TB in size and let ingestion time clustering automatically take effect. This feature will cluster the data based on the order the data was ingested by default for all tables.
- You can partition by a column if you expect data in each partition to be at least 1GB
- Always choose a low cardinality column — for example, year, date — as a partition column


In [0]:
# Partitioning during file write
df.write.mode("overwrite").format("delta").partitionBy("region").save("/Volumes/population_metrics/landing/datasets/output_dataset/delta_lake/countries_consolidated_partitioned")

In [0]:
# Partitioning during table write
df.write.partitionBy("region").saveAsTable("population_metrics.default.countries_consolidated_partitioned")

In [0]:
# Reading in the partitioned data
df1 = spark.read.format("delta").load("/Volumes/population_metrics/landing/datasets/output_dataset/delta_lake/countries_consolidated_partitioned")

In [0]:
# While this leverages the partitioned column, the overhead associated does not make the overall performance much better
# Partitioning yields true benefits on tables > 1TB
df1.filter("region = 'America'").display()

In [0]:
# Querying the non partitioned column as a comparison, due to the small file size the performance is only marginally slower
df1.filter("sub_region = 'Northern America'").display()